# Dataset Overview
Basic stats on the PTC corpus.

In [1]:
import sys
sys.path.append('..')

from src.data.corpus import load_corpus
from src.data.splits import make_splits
from collections import Counter
import numpy as np

In [2]:
train_all = load_corpus('train', '../data')
dev_unlabelled = load_corpus('dev', '../data')
train, dev, test = make_splits(train_all, seed=42)

print('=== Corpus size ===')
print(f'All labelled articles:    {len(train_all)}')
print(f'Unlabelled dev articles:  {len(dev_unlabelled)}')
print()
print('=== Our splits ===')
print(f'Train: {len(train)}')
print(f'Dev:   {len(dev)}')
print(f'Test:  {len(test)}')

=== Corpus size ===
All labelled articles:    371
Unlabelled dev articles:  75

=== Our splits ===
Train: 289
Dev:   41
Test:  41


In [3]:
# Article length distribution
lengths = [len(a.text) for a in train_all]

print('=== Article length (characters) ===')
print(f'Min:    {min(lengths)}')
print(f'Max:    {max(lengths)}')
print(f'Mean:   {np.mean(lengths):.0f}')
print(f'Median: {np.median(lengths):.0f}')

=== Article length (characters) ===
Min:    761
Max:    47473
Mean:   5612
Median: 4184


In [4]:
# Propaganda span counts
si_counts = [len(a.si_spans) for a in train_all]
articles_with_spans = sum(1 for c in si_counts if c > 0)

print('=== Propaganda spans per article (SI) ===')
print(f'Articles with no spans:   {sum(1 for c in si_counts if c == 0)}')
print(f'Articles with spans:      {articles_with_spans}')
print(f'Mean spans per article:   {np.mean(si_counts):.1f}')
print(f'Max spans in one article: {max(si_counts)}')

=== Propaganda spans per article (SI) ===
Articles with no spans:   14
Articles with spans:      357
Mean spans per article:   14.7
Max spans in one article: 192


In [5]:
# Technique distribution
technique_counts = Counter()
for a in train_all:
    for span in a.tc_spans:
        technique_counts[span.technique] += 1

total = sum(technique_counts.values())
print('=== Technique distribution (TC) ===')
print(f'{"Technique":<45} {"Count":>6}  {"% total":>8}')
print('-' * 65)
for technique, count in technique_counts.most_common():
    print(f'{technique:<45} {count:>6}  {count/total*100:>7.1f}%')
print(f'{"TOTAL":<45} {total:>6}')

=== Technique distribution (TC) ===
Technique                                      Count   % total
-----------------------------------------------------------------
Loaded_Language                                 2123     34.6%
Name_Calling,Labeling                           1058     17.3%
Repetition                                       621     10.1%
Doubt                                            493      8.0%
Exaggeration,Minimisation                        466      7.6%
Appeal_to_fear-prejudice                         294      4.8%
Flag-Waving                                      229      3.7%
Causal_Oversimplification                        209      3.4%
Appeal_to_Authority                              144      2.3%
Slogans                                          129      2.1%
Whataboutism,Straw_Men,Red_Herring               108      1.8%
Black-and-White_Fallacy                          107      1.7%
Thought-terminating_Cliches                       76      1.2%
Bandwagon,Reduct

In [6]:
# Span length distribution
span_lengths = []
for a in train_all:
    for span in a.si_spans:
        span_lengths.append(span.end - span.start)

print('=== Propaganda span length (characters) ===')
print(f'Total spans: {len(span_lengths)}')
print(f'Min:         {min(span_lengths)}')
print(f'Max:         {max(span_lengths)}')
print(f'Mean:        {np.mean(span_lengths):.0f}')
print(f'Median:      {np.median(span_lengths):.0f}')

=== Propaganda span length (characters) ===
Total spans: 5468
Min:         2
Max:         799
Mean:        50
Median:      25


In [7]:
# Class imbalance for SI: proportion of propaganda tokens
total_chars = sum(len(a.text) for a in train_all)
propaganda_chars = sum(span.end - span.start for a in train_all for span in a.si_spans)

print('=== Token-level imbalance (SI) ===')
print(f'Total characters:       {total_chars:,}')
print(f'Propaganda characters:  {propaganda_chars:,}  ({propaganda_chars/total_chars*100:.1f}%)')
print(f'Non-propaganda:         {total_chars-propaganda_chars:,}  ({(total_chars-propaganda_chars)/total_chars*100:.1f}%)')

=== Token-level imbalance (SI) ===
Total characters:       2,081,888
Propaganda characters:  275,776  (13.2%)
Non-propaganda:         1,806,112  (86.8%)
